# MiniT2I geral, treino no Colab

Este notebook treina do zero o gerador geral, o que entende qualquer prompt. Ele usa as fotos do CC3M (2,9 milhões de imagens com legenda), o VAE do Stable Diffusion e um encoder de texto CLIP multilíngue, os dois congelados. O transformer de 120 milhões de parâmetros é o que aprende.

Como usar

1. Ambiente de execução, Alterar tipo de ambiente, GPU T4.
2. Rode as células em ordem. A segunda pede permissão para usar o seu Google Drive, que é onde ficam os dados e o checkpoint.
3. A terceira célula prepara o dataset em segundo plano e a quarta treina ao mesmo tempo, usando os shards que já ficaram prontos.
4. Quando a sessão cair, abra de novo e rode tudo. Ele continua de onde parou, tanto o preparo quanto o treino.

A cada 30 minutos sai uma grade de exemplos na pasta minit2i/amostras do seu Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
PASTA = '/content/drive/MyDrive/minit2i'
DADOS = f'{PASTA}/dados'
os.makedirs(DADOS, exist_ok=True)
if os.path.exists('/content/minit2i'):
    !git -C /content/minit2i pull -q
else:
    !git clone -q https://github.com/Samuelpedroso-dev/minit2i /content/minit2i
!pip install -q diffusers sentence-transformers
print('pronto,', len(os.listdir(DADOS)), 'shards já preparados')

In [ ]:
# preparo do dataset em segundo plano, 200 shards, cerca de 1 milhão de imagens
log = open(f'{PASTA}/preparo.log', 'a')
prep = subprocess.Popen(['python', 'preparar.py', '--saida', DADOS, '--inicio', '0', '--fim', '200'],
                        cwd='/content/minit2i/geral', stdout=log, stderr=subprocess.STDOUT)
print('preparo rodando, acompanhe em minit2i/preparo.log')

In [ ]:
# treino, espera os primeiros 10 shards e começa. Salva a cada 15 minutos.
!cd /content/minit2i/geral && python treinar.py --dados {DADOS} --ckpt {PASTA}/geral.pt --minutos 600

In [ ]:
from IPython.display import Image, display
display(Image(f'{PASTA}/amostras/ultima.png', width=768))

In [ ]:
!tail -5 {PASTA}/preparo.log